# 🧠 Customer Sentiment Analysis with TensorFlow

Welcome! In this notebook, we will build a powerful **Deep Learning** model to analyze customer reviews. We'll classify them into **Positive**, **Negative**, or **Neutral** sentiments using a neural network built with **TensorFlow** and **Keras**.

### 🚀 Key Steps:
1.  **Data Loading**: Efficiently handling large datasets.
2.  **Preprocessing**: Cleaning text and preparing it for the neural network.
3.  **Model Building**: Creating an Embedding + LSTM architecture.
4.  **Training & Evaluation**: Teaching the model and checking its accuracy.
5.  **Interactive Prediction**: Enter your own reviews and see the AI in action!

## 1. Setup & Imports
First, we import the necessary libraries for data processing, natural language processing (NLP), and deep learning.

In [1]:
import pandas as pd
import numpy as np
import os
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

2025-12-20 21:43:49.637471: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-20 21:43:50.277816: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-20 21:43:54.776828: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Libraries imported successfully!
TensorFlow version: 2.20.0


## 2. Dataset Loading & Size Check
We load the dataset from the local `archive/Reviews.csv` file. To ensure smooth performance and fast training, we limit our analysis to the **first 50,000 rows** if the dataset is larger.

In [2]:
file_path = 'archive/Reviews.csv'
target_rows = 50000

if os.path.exists(file_path):
    # Check total size first
    total_rows = sum(1 for line in open(file_path)) - 1 # Minus header
    print(f"📊 Total rows found in dataset: {total_rows:,}")
    
    if total_rows > 80000:
        print(f"⚠️ Dataset is large. Automatically loading only the first {target_rows:,} rows for optimization.")
        df = pd.read_csv(file_path, nrows=target_rows)
    else:
        print("✅ Dataset size is manageable. Loading full data.")
        df = pd.read_csv(file_path)
    
    print(f"✅ Successfully loaded {len(df):,} rows.")
else:
    print(f"❌ Error: Dataset not found at {file_path}. Please check the folder.")

df.head()

📊 Total rows found in dataset: 568,454
⚠️ Dataset is large. Automatically loading only the first 50,000 rows for optimization.
✅ Successfully loaded 50,000 rows.


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## 3. Data Preprocessing
Before feeding data into our model, we need to tidy it up.

In [3]:
# 1. Select relevant columns
df = df[['Text', 'Score']]

# 2. Map Multi-class (Score 1-5) to 3 classes: Positive, Negative, Neutral
def map_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['Sentiment'] = df['Score'].apply(map_sentiment)

# 3. Clean product text
def clean_text(text):
    text = str(text).lower()                      # Lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)      # Remove special chars and numbers
    text = re.sub(r'\s+', ' ', text).strip()      # Remove extra whitespace
    return text

print("🧹 Cleaning text data...")
df['Cleaned_Text'] = df['Text'].apply(clean_text)

print("✅ Preprocessing complete. Sentiment distribution:")
print(df['Sentiment'].value_counts())
df.head()

🧹 Cleaning text data...
✅ Preprocessing complete. Sentiment distribution:
Sentiment
Positive    38418
Negative     7535
Neutral      4047
Name: count, dtype: int64


,Text,Score,Sentiment,Cleaned_Text
0,I have bought several of the Vitality canned d...,5,Positive,i have bought several of the vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...,1,Negative,product arrived labeled as jumbo salted peanut...
2,This is a confection that has been around a fe...,4,Positive,this is a confection that has been around a fe...
3,If you are looking for the secret ingredient i...,2,Negative,if you are looking for the secret ingredient i...
4,Great taffy at a great price. There was a wid...,5,Positive,great taffy at a great price there was a wide ...


### 3.1 Tokenization & Label Encoding
Computers understand numbers, not words. We convert our labels and text sentences into numerical formats.

In [4]:
# Label Encoding for Target (y)
encoder = LabelEncoder()
y = encoder.fit_transform(df['Sentiment'])
y = tf.keras.utils.to_categorical(y)

# Tokenization for Text (X)
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['Cleaned_Text'])

sequences = tokenizer.texts_to_sequences(df['Cleaned_Text'])
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Classes: {list(encoder.classes_)}")

Shape of X_train: (40000, 100)
Classes: ['Negative', 'Neutral', 'Positive']


## 4. Building the TensorFlow Model
We use an **Embedding** layer followed by an **LSTM** (Long Short-Term Memory) layer, which is excellent for understanding context in sequences like text.

In [5]:
embedding_dim = 128

model = Sequential([
    Embedding(max_words, embedding_dim, input_length=max_len),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax') # 3 output neurons for Positive, Negative, Neutral
])

model.compile(loss='categorical_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])

model.summary()

/home/likesh-kumar-sahu/anaconda3/envs/ml-cpu/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 5. Model Training & Evaluation
Now we train the model on our data. **Tip**: This might take a few minutes depending on your CPU/GPU.

In [6]:
epochs = 5
batch_size = 64

print("🚀 Starting training...")
history = model.fit(X_train, y_train, 
                    epochs=epochs, 
                    batch_size=batch_size, 
                    validation_data=(X_test, y_test), 
                    verbose=1)

print("✅ Training finished!")

# Evaluate on test set
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n📈 Final Test Accuracy: {acc*100:.2f}%")

🚀 Starting training...
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 118s 176ms/step - accuracy: 0.7691 - loss: 0.7021 - val_accuracy: 0.7631 - val_loss: 0.6872
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 107s 172ms/step - accuracy: 0.7804 - loss: 0.6250 - val_accuracy: 0.8140 - val_loss: 0.5480
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 106s 170ms/step - accuracy: 0.8303 - loss: 0.4739 - val_accuracy: 0.8328 - val_loss: 0.4653
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 106s 169ms/step - accuracy: 0.8559 - loss: 0.3944 - val_accuracy: 0.8389 - val_loss: 0.4534
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 110s 177ms/step - accuracy: 0.8715 - loss: 0.3461 - val_accuracy: 0.8316 - val_loss: 0.4562
✅ Training finished!

📈 Final Test Accuracy: 83.16%


## 6. User Input → Sentiment Prediction
Enter a review below to see how the model predicts it!

In [10]:
model.save('Sentiment_Analysis_Model.h5')


def predict_sentiment(user_text):
    # Preprocess input
    cleaned = clean_text(user_text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    
    # Predict
    prediction = model.predict(padded, verbose=0)
    class_idx = np.argmax(prediction)
    confidence = prediction[0][class_idx] * 100
    
    sentiment = encoder.inverse_transform([class_idx])[0]
    return sentiment, confidence

# --- TEST AREA ---
user_input = input("✍️ Enter a product review: ")

if user_input:
    label, conf = predict_sentiment(user_input)
    print(f"\n🔍 Model Result:")
    print(f"Sentiment: {label}")
    print(f"Confidence: {conf:.2f}%")
else:
    print("Please enter some text to test.")


🔍 Model Result:
Sentiment: Negative
Confidence: 73.13%
